In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
def inference(prompt: str) -> str:
    messages = [
        {"role": "system", "content": "You are Qwen. You are a helpful assistant who has access to knowledge from documents retrieved from vector database which is between <|context|>..knowledge..<|context|>."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


In [ ]:
prompt = "Explain traits in rust easily with example"
#inference(prompt)

'Certainly! Traits in Rust are a powerful feature that allow you to define behavior or interfaces that types can implement. They are similar to interfaces in other languages but have some unique characteristics in Rust. Here’s an easy-to-understand explanation along with examples.\n\n### What Are Traits?\nTraits in Rust are like contracts or specifications for what methods a type must provide. Types can implement these traits, and then you can use the methods defined in those traits as if they were part of the type itself.\n\n### Syntax\nA trait is defined using the `trait` keyword followed by the trait name, and then the methods that the trait defines. For example:\n\n```rust\ntrait MyTrait {\n    fn my_method(&self);\n}\n```\n\nThis `MyTrait` trait has one method, `my_method`, which takes a reference to `self` (the type implementing the trait).\n\n### Implementing Traits\nTo use a trait, you need to implement it for a specific type. This means you write the implementations of the met

In [5]:
import chromadb
from llama_index.core import PromptTemplate, Settings, SimpleDirectoryReader, StorageContext, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore

In [6]:
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
Settings.embed_model = embed_model

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:
from llama_index.llms.huggingface import HuggingFaceLLM
llm = HuggingFaceLLM(
    model=model,
    tokenizer=tokenizer,
)
Settings.llm = llm 

In [ ]:
documents = SimpleDirectoryReader(input_files=["./data/InternVL.pdf"]).load_data()
chroma_client = chromadb.EphemeralClient()
chroma_collection = chroma_client.create_collection("hf_db")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents, 
    storage_context=storage_context, 
    embed_model=embed_model,
    transformations=[SentenceSplitter(chunk_size=256, chunk_overlap=10)]
)


In [21]:
template = (
    "Imagine you are a research scientist's assistant and "
    "you answer supervisor's questions about AI research ."
    "Here is some context from Paper \n"
    "-----------------------------------------\n"
    "{context_str}\n"
    "-----------------------------------------\n"
    "Considering the above information, "
    "Please respond to the following inquiry:\n\n"
    "Question: {query_str}\n\n"
    "Answer succinctly and ensure your response is "
)
qa_template = PromptTemplate(template)

In [22]:
query_engine = index.as_query_engine(
    text_qa_template=qa_template,
    similarity_top_k=3
)

In [24]:
# Here I am uisng the InterVL2.5 paper
response = query_engine.query("what are the various parameter sizes these model come?")
print(response.response)

 supported by details from the paper:
The models mentioned in the paper have varying parameter sizes. For instance, InternViT, InternVL, and InternViT-Plus have been trained with 2B, 6B, and 13B parameters respectively. InternVL also includes InternViT, which has been scaled up to 22B parameters. Additionally, Molmo and pixmo models have been developed using open weights and open data, but specific parameter sizes are not detailed in the provided information. To summarize, InternViT was initially trained with 2B parameters and later scaled up to 22B parameters, while InternVL and InternViT-Plus were trained with 6B and 13B parameters respectively.
Based on the information provided in the paper, the models mentioned have varying parameter sizes as follows:

- InternViT was initially trained with 2B parameters and later scaled up to 22B parameters.
- InternVL includes InternViT and has been trained with 22B parameters.
- InternViT-Plus has been trained with 13B parameters.

So, the param